# Pipeline VLM Mamografía — Fine-Tuning MedGemma-4b-it
**Autor:** Luis Enrique Medrano Santana  
**Asesor:** Luis Vives Garnique  
**Institución:** Pontificia Universidad Católica del Perú (PUCP)  
**Título:** Modelo generativo multimodal de visión-lenguaje orientado al apoyo a la toma de decisiones clínicas en el diagnóstico de cáncer de mama

---
**Objetivo:** Fine-tuning multi-tarea de MedGemma 4B-IT sobre VinDr-Mammo para predecir simultáneamente:
- `density`: densidad mamaria ACR (A/B/C/D)
- `findings`: descripción textual de hallazgos por vista y lateralidad
- `birads`: categoría BI-RADS (1-5)


## 1. Instalación de dependencias

Versiones fijas para reproducibilidad. `bitsandbytes` habilita cuantización 4-bit NF4 para cargar MedGemma en ~5GB VRAM. `peft` provee LoRA. `trl` provee SFTTrainer.

In [ ]:
!pip install -q transformers==4.56.2 peft==0.18.1 trl==0.23.1 accelerate==1.10.1 \
    bitsandbytes==0.48.1 sentencepiece protobuf scikit-learn
print('Deps OK')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 150.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.6/564.6 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.8 MB/s eta 0:00:00
Deps OK


## 2. Montar Drive y definir rutas

- `PROCESSED_DIR`: PNGs preprocesados desde DICOM (Otsu ROI crop + CLAHE + resize 448x448, merge 896x896)
- `AUG_DIR`: PNGs augmentados por clase (flip horizontal + rotación ±15°) solo para BIRADS 3, 4, 5
- `CHECKPOINT_DIR`: donde se guardan los checkpoints por época
- `LOCAL_IMG_DIR` / `LOCAL_AUG_DIR`: disco local de Colab (mucho más rápido que Drive para I/O de training)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_BASE     = Path('/content/drive/MyDrive/vindr_1000')
METADATA_DIR   = DRIVE_BASE / 'metadata'

# Imagenes preprocesadas (DICOM -> Otsu -> CLAHE -> 448x448 -> merge 896x896)
PROCESSED_DIR  = DRIVE_BASE / 'processed_dcm'

# Augmentadas: subcarpetas BIRADS_3/, BIRADS_4/, BIRADS_5/
# Cada subcarpeta tiene: originales + _flip + _rot (flip prioritario, rot solo si falta)
AUG_DIR        = DRIVE_BASE / 'processed_aug_2'

# Checkpoints del modelo durante training
CHECKPOINT_DIR = DRIVE_BASE / 'checkpoints' / 'multitask_vf'

# Disco local Colab: mucho mas rapido que Drive para leer imagenes en cada step
LOCAL_IMG_DIR  = Path('/content/processed')
LOCAL_AUG_DIR  = Path('/content/processed_aug')

(CHECKPOINT_DIR / 'run').mkdir(parents=True, exist_ok=True)
print('Rutas OK')
print(f'  PROCESSED_DIR : {PROCESSED_DIR}')
print(f'  AUG_DIR       : {AUG_DIR}')
print(f'  CHECKPOINT_DIR: {CHECKPOINT_DIR}')

Mounted at /content/drive
Rutas OK
  PROCESSED_DIR : /content/drive/MyDrive/vindr_1000/processed_dcm
  AUG_DIR       : /content/drive/MyDrive/vindr_1000/processed_aug_2
  CHECKPOINT_DIR: /content/drive/MyDrive/vindr_1000/checkpoints/multitask_vf


## 3. Copiar imágenes a disco local + cache en RAM

**Por qué copiar a disco local:** Drive tiene ~50MB/s de lectura secuencial pero alta latencia por archivo. Durante training, el DataLoader lee ~1 imagen por step — con Drive esto introduce delays de 100-500ms por imagen que escalan a horas. Copiando a disco local primero, la lectura es instantánea.

**Cache en RAM:** Se pre-cargan todas las imágenes en `IMAGE_CACHE` (dict path→PIL). Elimina completamente el I/O durante training. Con 896x896 RGB, ~2200 imágenes ocupan ~5GB RAM — Colab Pro+ tiene 50GB+ disponibles.

In [ ]:
import subprocess
from PIL import Image as PILImage
from tqdm import tqdm
import pandas as pd

# --- Copiar processed_dcm a disco local ---
LOCAL_IMG_DIR.mkdir(exist_ok=True)
print('Copiando processed_dcm -> disco local...')
subprocess.run(['rsync', '-a', '--info=progress2',
    str(PROCESSED_DIR)+'/', str(LOCAL_IMG_DIR)+'/'], capture_output=False)

# --- Copiar augmentadas (estructura plana) a disco local ---
# Las augmentadas estan en subcarpetas BIRADS_3/, BIRADS_4/, BIRADS_5/
# Las copiamos a una carpeta plana para simplificar el indexado
LOCAL_AUG_DIR.mkdir(exist_ok=True)
print('Copiando augmentadas -> disco local (aplanando subcarpetas)...')
for b in [3, 4, 5]:
    subdir = AUG_DIR / f'BIRADS_{b}'
    if subdir.exists():
        subprocess.run(['rsync', '-a', str(subdir)+'/', str(LOCAL_AUG_DIR)+'/'],
                       capture_output=False)
        print(f'  BIRADS_{b}: copiado')

def resolve_path(p):
    """Busca primero en disco local, luego en aug local, finalmente usa ruta original."""
    name = Path(p).name
    local = LOCAL_IMG_DIR / name
    if local.exists(): return str(local)
    aug = LOCAL_AUG_DIR / name
    if aug.exists(): return str(aug)
    return str(p)

# --- Cache en RAM: cargar todas las imagenes procesadas ---
IMAGE_CACHE = {}
all_pngs = list(LOCAL_IMG_DIR.glob('*.png')) + list(LOCAL_AUG_DIR.glob('*.png'))
for p in tqdm(all_pngs, desc='Disco->RAM'):
    try:
        IMAGE_CACHE[str(p)] = PILImage.open(p).convert('RGB')
    except:
        pass

print(f'Cache RAM: {len(IMAGE_CACHE)} imagenes')

Copiando processed_dcm -> disco local...
Copiando augmentadas -> disco local (aplanando subcarpetas)...
  BIRADS_3: copiado
  BIRADS_4: copiado
  BIRADS_5: copiado


Disco->RAM: 100%|██████████| 5028/5028 [00:40<00:00, 124.17it/s]

Cache RAM: 5028 imagenes


## 4. Construir df_all — dataset base

Lee los dos CSVs de VinDr-Mammo:
- `breast-level_annotations.csv`: una fila por imagen (4 por estudio), con BIRADS y density por vista
- `finding_annotations.csv`: una fila por finding por imagen, con categoría del hallazgo

Para cada `study_id` que tenga PNG procesado, construye un registro con:
- `breast_birads`: BIRADS máximo entre las 4 vistas (el más severo)
- `density`: primer density_char encontrado (consistente entre vistas)
- `findings`: texto natural concatenando hallazgos por lateralidad+vista
- `report`: JSON string que será el target del modelo

**Nota:** Los estudios augmentados NO se agregan aquí — se incorporan en la celda de rebalanceo.

In [ ]:
import ast, json
import pandas as pd

breast_df  = pd.read_csv(METADATA_DIR / 'breast-level_annotations.csv')
finding_df = pd.read_csv(METADATA_DIR / 'finding_annotations.csv')

# Mapeos para texto natural en findings
LATERALITY_FULL = {'R': 'right', 'L': 'left'}
VIEW_FULL = {'CC': 'cranio-caudal (CC)', 'MLO': 'medio-lateral oblique (MLO)'}

def safe_last_alnum(s):
    """Extrae el ultimo caracter alfanumerico de un string (ej. 'BI-RADS 3' -> '3', 'DENSITY C' -> 'C')."""
    if s is None or (isinstance(s, float) and pd.isna(s)): return ''
    for ch in reversed(str(s).strip()):
        if ch.isalnum(): return ch
    return ''

def normalize_lat(x):
    """Normaliza lateralidad a 'L' o 'R'."""
    t = str(x).strip().upper() if x else ''
    return 'L' if t.startswith('L') else ('R' if t.startswith('R') else '')

def normalize_view(x):
    """Normaliza vista a 'CC' o 'MLO'."""
    t = str(x).strip().upper() if x else ''
    return 'MLO' if 'MLO' in t else ('CC' if 'CC' in t else '')

def parse_categories(raw):
    """Parsea finding_categories desde string a lista de strings."""
    if raw is None or (isinstance(raw, float) and pd.isna(raw)): return []
    try:
        out = ast.literal_eval(str(raw).strip())
        return [str(x).strip() for x in out] if isinstance(out, list) else [str(out).strip()]
    except:
        return [x.strip() for x in str(raw).strip('[]').replace("'",'').split(',') if x.strip()]

def build_findings(finding_df, study_id):
    """
    Construye el texto de findings y los flags binarios (mass/calcification/asymmetry)
    a partir de finding_annotations.csv para un study_id dado.
    Formato: '<finding> found in <laterality> <view>' separados por ' and '.
    Si no hay hallazgos: 'Healthy Breast. No Findings'.
    """
    frows = finding_df[finding_df['study_id'] == study_id]
    mass = calc = asym = 0
    if frows.empty:
        return 'Healthy Breast. No Findings', mass, calc, asym
    grouped, all_no_find = {}, True
    for _, row in frows.iterrows():
        lat, view = normalize_lat(row.get('laterality','')), normalize_view(row.get('view_position',''))
        for cat in parse_categories(row.get('finding_categories','[]')):
            cl = cat.lower().strip()
            if not cl: continue
            if 'mass' in cl: mass = 1
            if 'calcification' in cl: calc = 1
            if 'asymmetry' in cl: asym = 1
            if cl != 'no finding': all_no_find = False
            grouped.setdefault((lat,view), {}).setdefault(cat, 0)
            grouped[(lat,view)][cat] += 1
    if all_no_find:
        return 'Healthy Breast. No Findings', mass, calc, asym
    parts = []
    for (lat, view), cats in grouped.items():
        filtered = {c:n for c,n in cats.items() if c.lower() != 'no finding'}
        if not filtered: continue
        lf = LATERALITY_FULL.get(lat, lat.lower())
        vf = VIEW_FULL.get(view, view)
        phrase = ' and '.join(c.lower() for c in filtered)
        parts.append(f'{phrase} found in {lf} {vf}')
    return (' and '.join(parts) if parts else 'Healthy Breast. No Findings'), mass, calc, asym

# Indexar PNGs procesados por study_id
png_index = {p.stem: str(p) for p in LOCAL_IMG_DIR.glob('*.png')}
print(f'PNGs procesados encontrados: {len(png_index)}')

# Construir df_all: un registro por study_id que tenga PNG procesado
records = []
for study_id, sdf in tqdm(breast_df.groupby('study_id'), desc='Construyendo df_all'):
    img_path = png_index.get(study_id)
    if img_path is None: continue  # estudio sin PNG procesado
    birads_max, density_char = '', ''
    for _, r in sdf.iterrows():
        d = safe_last_alnum(r.get('breast_density'))
        b = safe_last_alnum(r.get('breast_birads'))
        if not density_char and d: density_char = d.upper()
        if b.isdigit() and (not birads_max or int(b) > int(birads_max)): birads_max = b
    findings, mass, calc, asym = build_findings(finding_df, study_id)
    # Suspicion: mapeo BIRADS -> 3 clases para clasificacion de sospecha
    suspicion = 'healthy' if birads_max=='1' else ('benign' if birads_max in {'2','3'} else 'suspicious')
    records.append({
        'study_id'     : study_id,
        'image_path'   : img_path,
        'breast_birads': f'BI-RADS {birads_max}',
        'split'        : sdf['split'].iloc[0],
        'report'       : json.dumps({
            'density': density_char, 'findings': findings,
            'birads': birads_max, 'mass': mass,
            'calcification': calc, 'asymmetry': asym,
            'suspicion': suspicion,
        })
    })

df_all = pd.DataFrame(records)
print(f'\ndf_all total: {len(df_all)} estudios')
print(df_all['breast_birads'].value_counts().sort_index())

PNGs procesados encontrados: 3828


Construyendo df_all: 100%|██████████| 5000/5000 [00:10<00:00, 496.81it/s]


df_all total: 3828 estudios
breast_birads
BI-RADS 1    1829
BI-RADS 2    1082
BI-RADS 3     436
BI-RADS 4     368
BI-RADS 5     113
Name: count, dtype: int64


## 5. Split train/val + rebalanceo con augmentadas

**Problema:** VinDr-Mammo está muy desbalanceado — BIRADS 1/2 dominan (~70% del train). Sin rebalanceo el modelo aprende a predecir siempre 1 o 2.

**Estrategia de rebalanceo:**
- BIRADS 1, 2: downsample a 500 (suficientes originales disponibles)
- BIRADS 3, 4: upsample usando augmentadas hasta 500 (originales + flips generados)
- BIRADS 5: upsample usando augmentadas hasta 200 (originales + flips + rotaciones)

**aug_index:** mapea cada `study_id` a sus versiones augmentadas (`_flip`, `_rot`, `_rotx`).
Los estudios augmentados heredan el mismo `report` JSON que el original porque la augmentación no cambia el diagnóstico.

**df_balanced:** resultado final — 2200 filas de train perfectamente balanceadas.

In [ ]:
import numpy as np
import re

# Separar train y val (test set de VinDr)
df_train_orig = df_all[df_all['split']=='training'].reset_index(drop=True)
val_df        = df_all[df_all['split']=='test'].reset_index(drop=True)

print(f'Train originales: {len(df_train_orig)}')
print(f'Val (test set):   {len(val_df)}')
print()

# --- Funcion para swap left<->right en findings ---
def swap_findings(report_json_str):
    """
    Para imagenes flipadas horizontalmente, invierte left<->right en el campo findings.
    El flip horizontal del merge 2x2 (R_CC|L_CC / R_MLO|L_MLO) produce
    (L_CC|R_CC / L_MLO|R_MLO), por lo que la mama que visualmente
    aparecia a la izquierda ahora aparece a la derecha y viceversa.
    """
    rep = json.loads(report_json_str)
    findings = rep.get('findings', '')
    if findings and findings != 'Healthy Breast. No Findings':
        # Swap usando placeholder para evitar doble reemplazo
        findings = re.sub(r'\bright\b', '__RIGHT__', findings, flags=re.IGNORECASE)
        findings = re.sub(r'\bleft\b', 'right', findings, flags=re.IGNORECASE)
        findings = findings.replace('__RIGHT__', 'left')
        rep['findings'] = findings
    return json.dumps(rep, ensure_ascii=False)

# --- Construir aug_index ---
aug_index = {}
for p in LOCAL_AUG_DIR.glob('*.png'):
    stem = p.stem
    study_id = re.sub(r'_(flip|rot|rotx)\d*$', '', stem)
    if study_id != stem:
        aug_index.setdefault(study_id, []).append(str(p))

print(f'Studies con augmentadas: {len(aug_index)}')
sample_sid = next(iter(aug_index))
print(f'Ejemplo: {sample_sid} -> {aug_index[sample_sid]}')
print()

# --- Rebalanceo ---
TARGET = {'BI-RADS 1':500, 'BI-RADS 2':500, 'BI-RADS 3':500, 'BI-RADS 4':500, 'BI-RADS 5':200}
balanced = []

for birads, target in TARGET.items():
    subset = df_train_orig[df_train_orig['breast_birads']==birads].copy()
    n = len(subset)
    print(f'{birads}: {n} originales -> target {target}')

    if n >= target:
        balanced.append(subset.sample(target, random_state=42))
        print(f'  -> Downsample a {target}')
    else:
        rows = [subset]
        needed = target - n

        aug_rows = []
        for _, row in subset.iterrows():
            sid = row['study_id']
            for aug_path in aug_index.get(sid, []):
                aug_row = row.copy()
                aug_row['image_path'] = aug_path
                # Si es flip, swap left<->right en findings del report
                if '_flip' in aug_path:
                    aug_row['report'] = swap_findings(row['report'])
                aug_rows.append(aug_row)

        if aug_rows:
            aug_df = pd.DataFrame(aug_rows)
            if len(aug_df) >= needed:
                rows.append(aug_df.sample(needed, random_state=42))
                print(f'  -> {n} orig + {needed} aug = {target}')
            else:
                rows.append(aug_df)
                still_needed = needed - len(aug_df)
                rows.append(subset.sample(still_needed, replace=True, random_state=42))
                print(f'  -> {n} orig + {len(aug_df)} aug + {still_needed} duplicados = {target}')
        else:
            rows.append(subset.sample(needed, replace=True, random_state=42))
            print(f'  -> {n} orig + {needed} duplicados = {target}')

        balanced.append(pd.concat(rows))

df_balanced = pd.concat(balanced).sample(frac=1, random_state=42).reset_index(drop=True)

aug_count  = df_balanced['image_path'].str.contains('_flip|_rot').sum()
orig_count = len(df_balanced) - aug_count

print(f'\n=== df_balanced ===')
print(df_balanced['breast_birads'].value_counts().sort_index())
print(f'Total: {len(df_balanced)}')
print(f'Originales: {orig_count} | Augmentadas: {aug_count}')

for p in tqdm(val_df['image_path'].tolist(), desc='Cache val'):
    rp = resolve_path(p)
    if rp not in IMAGE_CACHE:
        try: IMAGE_CACHE[rp] = PILImage.open(rp).convert('RGB')
        except: pass
print(f'Cache total: {len(IMAGE_CACHE)}')

Train originales: 3053
Val (test set):   775

Studies con augmentadas: 450
Ejemplo: a235482f737d9a3baa8313c8f4db16d6 -> ['/content/processed_aug/a235482f737d9a3baa8313c8f4db16d6_flip.png']

BI-RADS 1: 1468 originales -> target 500
  -> Downsample a 500
BI-RADS 2: 855 originales -> target 500
  -> Downsample a 500
BI-RADS 3: 345 originales -> target 500
  -> 345 orig + 155 aug = 500
BI-RADS 4: 295 originales -> target 500
  -> 295 orig + 205 aug = 500
BI-RADS 5: 90 originales -> target 200
  -> 90 orig + 110 aug = 200

=== df_balanced ===
breast_birads
BI-RADS 1    500
BI-RADS 2    500
BI-RADS 3    500
BI-RADS 4    500
BI-RADS 5    200
Name: count, dtype: int64
Total: 2200
Originales: 1730 | Augmentadas: 470


Cache val: 100%|██████████| 775/775 [00:00<00:00, 67779.84it/s]

Cache total: 5028


## 6. CONFIG — hiperparámetros del fine-tuning

**Run 6 — MammoWise multi-task paper replication** (paper: 0.6355 BIRADS acc a 10 épocas)

Cambios respecto a Run 5b:
- `lora_r=8` → código MammoWise (paper no especifica r)
- `lr_scheduler=linear` → paper/código MammoWise (Run5b usaba cosine)
- `max_grad_norm=0.3` → código MammoWise (Run5b=1.0)
- `warmup_ratio=0.03` → código MammoWise (Run5b=0.05)
- `label_smoothing=0.0` → MammoWise no usa label_smoothing
- `es_patience=6` → más paciencia para llegar a época 10 donde paper reporta mejor resultado
- **LoRA solo en attention** (q/k/v/o, sin FFN) → código MammoWise, menos params, menos overfitting
- **Collate sin masking de prompt** → código MammoWise, loss sobre secuencia completa

Igual al paper: `alpha=16, dropout=0.05, lr=2e-4, batch=1, adamw_bnb_8bit, grad_accum=8`

In [ ]:
CONFIG = {
    'model_id'           : 'google/medgemma-4b-it',
    'lora_r'             : 16,       # Run 1: scaling alpha/r=1.0 (magnitud MammoWise)
    'lora_alpha'         : 16,       # Paper MammoWise: alpha=16
    'lora_dropout'       : 0.05,     # Paper MammoWise: dropout=0.05
    'lr'                 : 2e-4,     # Paper MammoWise: lr=2e-4 (cambio principal vs Run 4)
    'epochs'             : 10,       # Run 1: overfit confirmado tras epoca 10
    'batch_size'         : 1,        # Paper MammoWise: batch=1
    'grad_accum'         : 8,        # Paper MammoWise: grad_accum=8
    'max_grad_norm'      : 0.3,      # codigo MammoWise
    'warmup_ratio'       : 0.03,     # codigo MammoWise
    'weight_decay'       : 0.01,
    'max_new_tokens_val' : 124,
    'val_n_per_class'    : 15,
    'seed'               : 20201531,
    'es_patience'        : 6,        # EarlyStopping basado en BIRADS F1 macro
    'label_smoothing'    : 0.0,      # MammoWise no usa label_smoothing
}
print(CONFIG)

## 7. Prompt del sistema + targets + parsers

**SYSTEM_PROMPT:** rol del modelo (radiologo certificado, meticulos, reporte clínico).

**USER_PROMPT:** instrucciones completas incluyendo:
- Layout del merge 2x2 (top-left=L_CC, top-right=R_CC, bot-left=L_MLO, bot-right=R_MLO)
- Clasificación ACR de densidad (A/B/C/D)
- Formato de findings con lateralidad y vista
- Escala BI-RADS (1-5)
- Instrucción de output: SOLO JSON válido, sin markdown

**build_target():** extrae del report JSON solo los campos que el modelo debe predecir (density, findings, birads) — omite mass/calcification/asymmetry/suspicion del target de training.

**parse_birads / parse_density:** extractores robustos del output generado por el modelo. Manejan casos donde el modelo genera texto extra o formatos ligeramente distintos.

In [ ]:
import torch, re, json

SYSTEM_PROMPT = (
    'You are a board-certified breast radiologist with extensive experience in screening '
    'mammography. You are meticulous and produce clear, clinically actionable reports.'
)

USER_PROMPT = (
    'Interpret the mammogram and produce a structured report.\n\n'
    'Image layout (2x2 composite):\n'
    '  top-left  = RIGHT breast, cranio-caudal (CC) view\n'
    '  top-right = LEFT breast, cranio-caudal (CC) view\n'
    '  bot-left  = RIGHT breast, medio-lateral oblique (MLO) view\n'
    '  bot-right = LEFT breast, medio-lateral oblique (MLO) view\n\n'
    'Breast density (ACR): A almost entirely fatty, B scattered fibroglandular, '
    'C heterogeneously dense, D extremely dense.\n\n'
    'Findings: describe abnormalities with laterality and view. '
    'Possible findings: mass, suspicious calcification, focal asymmetry, '
    'architectural distortion, asymmetry, suspicious lymph node, skin thickening, '
    'nipple retraction, global asymmetry, skin retraction. '
    'Format: "<finding> found in <left|right> <cranio-caudal (CC)|medio-lateral oblique (MLO)>". '
    'Multiple findings separated by " and ". '
    'If none: "Healthy Breast. No Findings".\n\n'
    'BI-RADS overall assessment: 1 negative, 2 benign, 3 probably benign, '
    '4 suspicious, 5 highly suggestive of malignancy.\n\n'
    'Return ONLY valid JSON. No markdown. No explanation. No text outside the JSON.\n'
    'Use exactly this schema (BI-RADS first):\n'
    '{"birads": "BIRADS <1-5> - <assessment description>", '
    '"density": "Density <A-D> - <description>", '
    '"findings": "<description>"}'
)

# Descripciones deterministas (estilo MammoWise) para enriquecer birads y density
BIRADS_DESCRIPTIONS = {
    '1': 'Negative. Healthy breast.',
    '2': 'Benign finding.',
    '3': 'Probably benign finding. Short-term follow-up is suggested.',
    '4': 'Suspicious abnormality. Biopsy should be considered.',
    '5': 'Highly suggestive of malignancy. Appropriate action should be taken.',
}
DENSITY_DESCRIPTIONS = {
    'A': 'Almost entirely fatty.',
    'B': 'Scattered areas of fibroglandular density.',
    'C': 'Heterogeneously dense; may obscure small masses.',
    'D': 'Extremely dense; lowers the sensitivity of mammography.',
}

def build_target(row):
    """Target descriptivo (Run 1): birads y density con descripcion + findings. BIRADS primero."""
    rep = json.loads(row['report'])
    b = str(rep['birads']).strip()
    d = str(rep['density']).strip().upper()
    return json.dumps({
        'birads'  : f'BIRADS {b} - {BIRADS_DESCRIPTIONS.get(b, "")}'.strip(),
        'density' : f'Density {d} - {DENSITY_DESCRIPTIONS.get(d, "")}'.strip(),
        'findings': rep['findings'],
    }, ensure_ascii=False)

def parse_birads(text):
    """Extrae BIRADS (1-5) del output. Robusto al formato descriptivo 'BIRADS 3 - ...'."""
    val = text
    try:
        d = json.loads(text)
        val = str(d.get('birads', ''))
    except: pass
    m = re.search(r'BIRADS\s*([1-5])', val, re.IGNORECASE)
    if m: return m.group(1)
    m = re.search(r'([1-5])', val)
    if m: return m.group(1)
    m = re.search(r'"birads"\s*:\s*"[^"]*?([1-5])', text)
    return m.group(1) if m else 'UNKNOWN'

def parse_density(text):
    """Extrae density (A/B/C/D) del output. Robusto al formato descriptivo 'Density C - ...'."""
    val = text
    try:
        d = json.loads(text)
        val = str(d.get('density', ''))
    except: pass
    m = re.search(r'Density\s*([A-D])', val, re.IGNORECASE)
    if m: return m.group(1).upper()
    m = re.search(r'\b([A-D])\b', val)
    if m: return m.group(1).upper()
    m = re.search(r'"density"\s*:\s*"[^"]*?([A-D])', text)
    return m.group(1).upper() if m else 'UNKNOWN'

print('Parsers OK (formato descriptivo)')
print('\n5 ejemplos de targets (verificacion):')
for i in [0, 200, 500, 1000, 1500]:
    if i < len(df_balanced):
        row = df_balanced.iloc[i]
        print(f'  {row["breast_birads"]} | {build_target(row)}')

In [ ]:
# === Exportar el nuevo target descriptivo a CSV (Run 1) ===
# El modelo se entrena con build_target() al vuelo; aqui lo persistimos para inspeccion/tesis.
import pandas as pd

def _row_to_record(row):
    return {
        'study_id'     : row['study_id'],
        'image_path'   : row['image_path'],
        'breast_birads': row['breast_birads'],
        'split'        : row.get('split', ''),
        'target'       : build_target(row),   # JSON descriptivo que realmente ve el modelo
    }

train_targets = pd.DataFrame([_row_to_record(r) for _, r in df_balanced.iterrows()])
val_targets   = pd.DataFrame([_row_to_record(r) for _, r in val_df.iterrows()])

out_dir = CHECKPOINT_DIR / 'run'
out_dir.mkdir(parents=True, exist_ok=True)
train_csv = out_dir / 'run1_targets_train.csv'
val_csv   = out_dir / 'run1_targets_val.csv'
train_targets.to_csv(train_csv, index=False)
val_targets.to_csv(val_csv, index=False)

print(f'Train targets: {len(train_targets)} -> {train_csv}')
print(f'Val targets:   {len(val_targets)} -> {val_csv}')
print('\nEjemplo de target nuevo:')
print(train_targets.iloc[0]['target'])

## 8. Login HuggingFace + Processor

MedGemma es un modelo de acceso restringido — requiere login con token HF que tenga acceso aprobado a `google/medgemma-4b-it`.

El `AutoProcessor` maneja tanto el texto (tokenizer) como las imágenes (SigLIP preprocessing). `padding_side='right'` es requerido durante training (para que el padding no interfiera con el loss masking).

In [ ]:
from huggingface_hub import login
from transformers import AutoProcessor
from google.colab import userdata

# Login con token HF almacenado en Colab Secrets (no hardcodear el token)
login(token=userdata.get('HF_TOKEN'))

processor = AutoProcessor.from_pretrained(CONFIG['model_id'], use_fast=True)
processor.tokenizer.padding_side = 'right'  # requerido para CLM training

# Detectar el ID del token de imagen especial de MedGemma
# Necesario para maskear esos tokens en el calculo de loss
IMAGE_TOKEN_ID = None
for tok in ['<image_soft_token>','<img>','<image>']:
    tid = processor.tokenizer.convert_tokens_to_ids(tok)
    if tid and tid != processor.tokenizer.unk_token_id:
        IMAGE_TOKEN_ID = tid; break
if IMAGE_TOKEN_ID is None: IMAGE_TOKEN_ID = 262144  # fallback conocido para MedGemma

print(f'IMAGE_TOKEN_ID: {IMAGE_TOKEN_ID}')
print('Processor OK')

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

IMAGE_TOKEN_ID: 262144
Processor OK


## 9. Dataset + Stratified Batch Sampler

**VinDrDataset:** PyTorch Dataset que retorna imagen PIL + mensajes en formato chat de MedGemma.

**StratifiedBatchSampler:** garantiza que cada batch efectivo (grad_accum pasos) contenga ejemplos de TODAS las 5 clases de BIRADS.

**Por qué es necesario:** VinDr tiene correlación casi perfecta findings→BIRADS en el texto sintético. Sin sampler, batches consecutivos dominados por BIRADS 1/2 refuerzan el prior del modelo. El sampler forza al modelo a discriminar entre las 5 clases desde el step 1, previniendo colapso a las clases mayoritarias.

**Implementación:** permuta indices por clase al inicio de cada época, luego toma `n_per_class` de cada clase por batch. `n_per_class = batch_size // n_classes = 8//5 = 1`.

In [ ]:
from torch.utils.data import Dataset, Sampler
import numpy as np

class VinDrDataset(Dataset):
    """Dataset de VinDr-Mammo para fine-tuning de MedGemma."""
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = resolve_path(row['image_path'])
        # Usar cache RAM si disponible, sino cargar desde disco
        img      = IMAGE_CACHE.get(img_path) or PILImage.open(img_path).convert('RGB')
        target   = build_target(row)
        # Formato de mensajes chat: usuario (prompt+imagen) -> asistente (JSON target)
        messages = [
            {'role':'user','content':[
                {'type':'text','text': SYSTEM_PROMPT+'\n\n'+USER_PROMPT},
                {'type':'image','image': img},
            ]},
            {'role':'assistant','content':[{'type':'text','text': target}]},
        ]
        return {'messages': messages, 'image': img, 'target_text': target}

class StratifiedBatchSampler(Sampler):
    """
    Sampler estratificado que garantiza representacion de todas las clases
    en cada batch efectivo. Cada batch contiene n_per_class ejemplos por clase.
    Numero de batches limitado por la clase con menos ejemplos.
    """
    def __init__(self, labels, batch_size, seed=42):
        self.labels      = np.array(labels)
        self.classes     = np.unique(self.labels)
        self.n_per_class = max(1, batch_size // len(self.classes))
        self.seed        = seed

    def __iter__(self):
        rng = np.random.default_rng(self.seed)
        # Permutacion aleatoria de indices por clase al inicio de cada epoca
        class_idx = {
            c: rng.permutation(np.where(self.labels==c)[0]).tolist()
            for c in self.classes
        }
        ptrs = {c: 0 for c in self.classes}
        # Numero de batches limitado por la clase minoritaria
        n_batches = min(len(idx)//self.n_per_class for idx in class_idx.values())
        for _ in range(n_batches):
            batch = []
            for c in self.classes:
                p = ptrs[c]
                batch.extend(class_idx[c][p:p+self.n_per_class])
                ptrs[c] += self.n_per_class
            rng.shuffle(batch)  # mezclar clases dentro del batch
            yield batch

    def __len__(self):
        return min((self.labels==c).sum()//self.n_per_class for c in self.classes)

# Construir datasets
birads_labels = df_balanced['breast_birads'].str.extract(r'(\d)')[0].astype(int).values
batch_eff     = CONFIG['batch_size'] * CONFIG['grad_accum']  # batch efectivo = 8

train_ds = VinDrDataset(df_balanced)
val_ds   = VinDrDataset(
    val_df.groupby('breast_birads', group_keys=False).apply(
        lambda x: x.sample(min(CONFIG['val_n_per_class'], len(x)), random_state=42),
        include_groups=False
    ).reset_index(drop=True)
)
sampler = StratifiedBatchSampler(birads_labels, batch_size=batch_eff, seed=CONFIG['seed'])

print(f'Train dataset: {len(train_ds)} muestras')
print(f'Val dataset:   {len(val_ds)} muestras ({CONFIG["val_n_per_class"]} por clase)')
print(f'Batches/epoca: {len(sampler)} | batch efectivo: {batch_eff}')
print(f'n_per_class por batch: {sampler.n_per_class}')
print('\nVerificacion primeros 3 batches (clases representadas):')
for i, b in enumerate(sampler):
    if i >= 3: break
    print(f'  Batch {i}: clases {sorted([birads_labels[j] for j in b])}')

Train dataset: 2200 muestras
Val dataset:   75 muestras (15 por clase)
Batches/epoca: 200 | batch efectivo: 8
n_per_class por batch: 1

Verificacion primeros 3 batches (clases representadas):
  Batch 0: clases [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
  Batch 1: clases [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
  Batch 2: clases [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


## 10. Collate — preparacion de batches

**Run 6 (MammoWise style):** loss sobre la secuencia completa (prompt + respuesta).
MammoWise no enmascara el prompt — el modelo computa loss en todos los tokens.

Se enmascaran solo:
1. Tokens de padding
2. Tokens de imagen (`IMAGE_TOKEN_ID`)
3. Token especial BOI (begin-of-image) si existe

**Sin truncación:** inputs ~700 tokens caben en el contexto de 8192 tokens del modelo.

In [ ]:
from functools import partial

# IDs del marcador de inicio del turno del asistente (Gemma): "<start_of_turn>model\n"
MODEL_TURN_IDS = processor.tokenizer.encode('<start_of_turn>model\n', add_special_tokens=False)

def collate_fn(examples, processor, image_token_id, model_turn_ids):
    # Run 1: loss SOLO sobre el JSON target (masking assistant-only del prompt)
    texts, images = [], []
    for ex in examples:
        images.append([ex['image']])
        chat = processor.apply_chat_template(
            ex['messages'], add_generation_prompt=False, tokenize=False).strip()
        texts.append(chat)

    batch  = processor(text=texts, images=images, return_tensors='pt', padding='longest')
    labels = batch['input_ids'].clone()

    # 1. Maskear padding
    pad_id = processor.tokenizer.pad_token_id
    if pad_id is not None: labels[labels==pad_id] = -100
    # 2. Maskear tokens de imagen
    if image_token_id is not None: labels[labels==image_token_id] = -100
    # 3. Maskear token BOI (begin-of-image)
    boi = processor.tokenizer.special_tokens_map.get('boi_token')
    if boi:
        bid = processor.tokenizer.convert_tokens_to_ids(boi)
        if bid and bid != processor.tokenizer.unk_token_id:
            labels[labels==bid] = -100

    # 4. Masking assistant-only: -100 a todo lo anterior (e incluyendo) el marcador del turno del modelo
    mt  = torch.tensor(model_turn_ids, dtype=batch['input_ids'].dtype)
    L   = len(model_turn_ids)
    ids = batch['input_ids']
    for r in range(ids.shape[0]):
        seq   = ids[r]
        found = None
        for j in range(seq.shape[0] - L + 1):
            if torch.equal(seq[j:j+L], mt):
                found = j + L  # primer token del target (tras el marcador)
        if found is not None:
            labels[r, :found] = -100
        # si no se encuentra (no deberia): queda el masking de pad/imagen

    batch['labels'] = labels
    return batch

collate = partial(collate_fn, processor=processor,
                  image_token_id=IMAGE_TOKEN_ID, model_turn_ids=MODEL_TURN_IDS)

# Sanity check: con masking assistant-only, los tokens activos deben ser SOLO el JSON target
b = collate([train_ds[0], train_ds[1]])
n_active = (b['labels'] != -100).sum().item()
print(f'Tokens activos en loss (2 muestras): {n_active} (~{n_active//2} por muestra)')
print(f'Shape input_ids: {b["input_ids"].shape}')
print('Tokens activos decodificados (deben ser SOLO el JSON target):')
for i in range(2):
    active  = b['input_ids'][i][b['labels'][i] != -100]
    decoded = processor.tokenizer.decode(active)
    print(f'  Muestra {i}: {repr(decoded)}')
print('Collate OK (masking assistant-only)')

## 11. Cargar modelo + LoRA

**Cuantizacion 4-bit NF4:** reduce MedGemma 4B de ~16GB a ~5GB VRAM.

**LoRA target modules (Run 6):** solo capas de atención del LM (q/k/v/o_proj), excluyendo FFN (gate/up/down_proj) y vision encoder. Codigo MammoWise usa `["q_proj","k_proj","v_proj","o_proj"]`. Con 2200 muestras, menos parametros entrenables reduce overfitting.

**modules_to_save=['lm_head', 'embed_tokens'] + ensure_weight_tying=True:** Gemma ata lm_head.weight == embed_tokens.weight. PEFT rompe este tie silenciosamente (PEFT issue #2864). Se mantiene por correctitud aunque MammoWise no lo incluye.

In [ ]:
from transformers import AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import re as _re

torch_dtype = torch.bfloat16

# Configuracion de cuantizacion 4-bit NF4 (QLoRA)
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',            # Normal Float 4 - mejor para pesos pre-entrenados
    bnb_4bit_compute_dtype=torch_dtype,   # bfloat16 para calculos
    bnb_4bit_use_double_quant=True,       # doble cuantizacion para mas ahorro
)
print(f'GPU: {torch.cuda.get_device_name(0)}')

# Cargar modelo base cuantizado
model = AutoModelForImageTextToText.from_pretrained(
    CONFIG['model_id'],
    quantization_config=bnb,
    device_map='auto',
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    attn_implementation='eager',  # 'eager' requerido para gradient checkpointing con 4-bit
)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model.config.use_cache = False    # incompatible con gradient checkpointing
model.enable_input_require_grads()  # necesario para LoRA con gradient checkpointing

# Run 1: LoRA en TODOS los linear del LM (attn + FFN), sin vision encoder
lm_modules = [
    n for n,_ in model.named_modules()
    if _re.match(
        r'^(?!.*vision).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$', n
    )
]
print(f'LoRA target modules: {len(lm_modules)} capas')

# Configuracion LoRA
peft_cfg = LoraConfig(
    r               = CONFIG['lora_r'],
    lora_alpha      = CONFIG['lora_alpha'],
    lora_dropout    = CONFIG['lora_dropout'],
    bias            = 'none',
    target_modules  = lm_modules,
    task_type       = 'CAUSAL_LM',
    modules_to_save     = ['lm_head', 'embed_tokens'],  # Run 4: fix weight tying (PEFT #2864)
    ensure_weight_tying = True,                          # Gemma: lm_head.weight == embed_tokens.weight
)
model = get_peft_model(model, peft_cfg)

def align_dtypes(model, dtype=torch.bfloat16):
    """Alinea parametros entrenables a dtype para evitar errores con 4-bit quantization."""
    n = 0
    for p in model.parameters():
        if p.requires_grad and p.dtype in (torch.float32, torch.float16):
            p.data = p.data.to(dtype)
            n += 1
    print(f'Aligned {n} params -> {dtype}')

align_dtypes(model)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Parametros entrenables: {trainable:,} ({100*trainable/total:.2f}%)')
print(f'VRAM usada: {torch.cuda.memory_allocated()/1e9:.1f}GB')

## 12. Funcion de validacion generativa

**Por qué validacion generativa en vez de val_loss:** la loss de validacion mide perplexity del texto generado, no accuracy diagnostica. Un modelo puede tener baja loss pero predecir siempre el mismo BIRADS.

**validate_custom:** genera respuestas reales con `model.generate()` y extrae BIRADS+density con los parsers. Reporta accuracy y F1 macro por clase.

**temperature=0.1 (AMRG):** generacion casi determinista para evaluacion reproducible. `do_sample=True` con temperatura muy baja es equivalente a greedy pero mas estable.

**Detalles de implementacion:** se desactiva gradient checkpointing y se activa `use_cache=True` durante generacion (requerido para `model.generate()`), luego se restaura para training.

In [ ]:
from sklearn.metrics import f1_score

device = torch.device('cuda')

def validate_custom(model, val_df, processor, device,
                    n_per_class=15, max_new_tokens=124, show_examples=5):
    """
    Validacion generativa: genera respuestas reales y mide accuracy/F1 de BIRADS y density.
    Mas costosa que val_loss pero refleja el rendimiento diagnostico real.
    """
    # Preparar modelo para inferencia
    model.eval()
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    orig_pad = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = 'left'  # requerido para generacion

    # Muestra estratificada del val set
    val_small = val_df.groupby('breast_birads', group_keys=False).apply(
        lambda x: x.sample(min(n_per_class, len(x)), random_state=42),
        include_groups=False
    ).reset_index(drop=True)

    preds, refs, dens_preds, dens_refs = [], [], [], []
    for idx, (_, row) in enumerate(val_small.iterrows()):
        img_path = resolve_path(row['image_path'])
        img = IMAGE_CACHE.get(img_path) or PILImage.open(img_path).convert('RGB')
        msgs = [{'role':'user','content':[
            {'type':'text','text': SYSTEM_PROMPT+'\n\n'+USER_PROMPT},
            {'type':'image','image': img}]}]
        text   = processor.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        inputs = processor(text=text, images=[img], return_tensors='pt').to(device)
        inputs = {k: v.to(torch.bfloat16) if torch.is_floating_point(v) else v
                  for k,v in inputs.items()}
        with torch.no_grad():
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                gen = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                )
        # Decodificar solo los tokens nuevos (excluir el prompt)
        dec = processor.tokenizer.decode(
            gen[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
        del inputs, gen; torch.cuda.empty_cache()

        rep   = json.loads(row['report'])
        p_bir = parse_birads(dec)
        p_den = parse_density(dec)

        if idx < show_examples:
            print(f"\n{'='*50}")
            print(f"GT birads={rep['birads']} density={rep['density']}")
            print(f"OUTPUT: {repr(dec)}")
            print(f"Parsed -> birads={p_bir} {'OK' if p_bir==rep['birads'] else 'FAIL'} | "
                  f"density={p_den} {'OK' if p_den==rep['density'] else 'FAIL'}")

        preds.append(p_bir);      refs.append(str(rep['birads']))
        dens_preds.append(p_den); dens_refs.append(rep['density'])

    # Restaurar modelo para training
    processor.tokenizer.padding_side = orig_pad
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    model.train()

    # Metricas globales
    n     = len(refs)
    b_acc = sum(p==r for p,r in zip(preds,refs))/n
    d_acc = sum(p==r for p,r in zip(dens_preds,dens_refs))/n
    b_f1  = f1_score(refs, preds, average='macro', zero_division=0)
    d_f1  = f1_score(dens_refs, dens_preds, average='macro', zero_division=0)

    def _pm1(p, r):
        try: return abs(int(p) - int(r)) <= 1
        except: return False
    b_acc_pm1 = sum(_pm1(p, r) for p, r in zip(preds, refs)) / n

    print(f'\n  BI-RADS acc={b_acc:.4f} (+/-1={b_acc_pm1:.4f}) F1={b_f1:.4f} | Density acc={d_acc:.4f} F1={d_f1:.4f}')
    for c in ['1','2','3','4','5']:
        rs = [r for r in refs if r==c]
        if rs:
            ps = [p for p,r in zip(preds,refs) if r==c]
            print(f'    BI-RADS {c}: acc={sum(pp==rr for pp,rr in zip(ps,rs))/len(rs):.4f} n={len(rs)}')

    return {'birads_accuracy':b_acc,'birads_accuracy_pm1':b_acc_pm1,'birads_f1':b_f1,
            'density_accuracy':d_acc,'density_f1':d_f1,'n_total':n}

print('validate_custom OK')

## 13. Callbacks — colapso, early stopping y loss

**CollapseCallback:** evalúa cada `check_every` steps sobre 3 muestras por clase (15 total).
- Detecta colapso si >60% de predicciones caen en una clase.
- Calcula **BIRADS F1 macro** y aplica **early stopping**: para el training si no hay mejora
  en `es_patience` evaluaciones consecutivas (4 × 200 steps ≈ 3 épocas).
  Evita el overfitting post-época 10 documentado en Run 1 y MammoWise.

**StepLossCallback:** imprime loss, learning rate y grad_norm en cada logging step.

**MultiTaskCDWTrainer** (celda siguiente): SFTTrainer extendido con:
- **CDW-CE:** penalidad ordinal en tokens BIRADS — predecir BIRADS 5 cuando la verdad
  es BIRADS 1 recibe pena >> predecir BIRADS 2. Mejora discriminación BIRADS 3/4/5.
- **Task weighting:** tokens BIRADS reciben peso `w_birads=1.5` en el loss (prioridad diagnóstica).

In [ ]:
from transformers import TrainerCallback
from collections import Counter
from sklearn.metrics import f1_score as _f1_score

class CollapseCallback(TrainerCallback):
    '''
    Monitorea colapso de predicciones y aplica early stopping basado en BIRADS F1 macro.
    Evalua cada check_every steps sobre 3 muestras por clase (15 total, fijas).
    Para el entrenamiento si F1 macro BIRADS no mejora en es_patience checks consecutivos.
    '''
    def __init__(self, val_df, processor, device, check_every=200, es_patience=4):
        self.val_fixed = val_df.groupby('breast_birads').apply(
            lambda x: x.sample(min(3, len(x)), random_state=42), include_groups=False
        ).reset_index(drop=True)
        self.processor   = processor
        self.device      = device
        self.check_every = check_every
        self.history     = []
        self.best_f1     = 0.0
        self.no_improve  = 0
        self.patience    = es_patience

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step % self.check_every != 0 or state.global_step == 0: return
        model.eval(); model.gradient_checkpointing_disable()
        model.config.use_cache = True
        orig = self.processor.tokenizer.padding_side
        self.processor.tokenizer.padding_side = 'left'
        preds, refs, outputs = [], [], []
        with torch.no_grad():
            for _, row in self.val_fixed.iterrows():
                try:
                    img = IMAGE_CACHE.get(resolve_path(row['image_path'])) or \
                          PILImage.open(resolve_path(row['image_path'])).convert('RGB')
                    msgs = [{'role':'user','content':[
                        {'type':'text','text': SYSTEM_PROMPT+'\n\n'+USER_PROMPT},
                        {'type':'image','image': img}]}]
                    text = self.processor.apply_chat_template(
                        msgs, add_generation_prompt=True, tokenize=False)
                    inputs = self.processor(text=text, images=[img], return_tensors='pt')
                    inputs = {k: v.to(device=self.device, dtype=torch.bfloat16)
                              if torch.is_floating_point(v)
                              else v.to(device=self.device)
                              for k, v in inputs.items()}
                    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                        gen = model.generate(**inputs, max_new_tokens=124, do_sample=False)
                    dec = self.processor.tokenizer.decode(
                        gen[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
                    rep = json.loads(row['report'])
                    preds.append(parse_birads(dec))
                    refs.append(str(rep['birads']))
                    outputs.append({'gt_birads': rep['birads'], 'gt_density': rep['density'], 'raw': dec})
                    del inputs, gen
                except Exception as e:
                    preds.append('UNKNOWN'); refs.append('?')
                    outputs.append({'gt_birads':'?', 'gt_density':'?', 'raw': f'ERROR: {e}'})

        torch.cuda.empty_cache()
        self.processor.tokenizer.padding_side = orig
        model.config.use_cache = False
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        model.train()

        c = Counter(preds)
        dom, cnt = c.most_common(1)[0]
        pct    = cnt / len(preds)
        status = f'COLAPSO -> "{dom}" {pct:.0%}' if pct > 0.6 else 'OK'
        per_class = {}
        for b in ['1','2','3','4','5']:
            gt_idx = [i for i, r in enumerate(refs) if r == b]
            if gt_idx:
                correct = sum(1 for i in gt_idx if preds[i] == b)
                per_class[b] = f'{correct}/{len(gt_idx)}'

        valid_refs  = [r for r, p in zip(refs, preds) if r != '?']
        valid_preds = [p for r, p in zip(refs, preds) if r != '?']
        curr_f1 = _f1_score(valid_refs, valid_preds, average='macro', zero_division=0) if valid_refs else 0.0

        print(f'\n  [step {state.global_step}] dist={dict(c)} | {status}')
        print(f'  Acc por clase: {per_class} | BIRADS F1_macro={curr_f1:.4f}')
        print(f'  --- Outputs (primeros 5) ---')
        for o, p in list(zip(outputs, preds))[:5]:
            marker = 'OK' if p == o['gt_birads'] else 'FAIL'
            print(f'  GT birads={o["gt_birads"]} density={o["gt_density"]}')
            print(f'  PRED: {o["raw"][:200]}')
            print(f'  -> birads parsed={p} {marker}')
            print()

        self.history.append({
            'step': state.global_step, 'dist': dict(c),
            'per_class': per_class, 'f1_macro': curr_f1,
        })

        # Early stopping basado en BIRADS F1 macro
        if curr_f1 > self.best_f1 + 0.005:  # umbral minimo de mejora
            self.best_f1 = curr_f1
            self.no_improve = 0
            print(f'  [ES] Nuevo mejor F1: {self.best_f1:.4f}')
        else:
            self.no_improve += 1
            print(f'  [ES] Sin mejora {self.no_improve}/{self.patience} | mejor F1={self.best_f1:.4f}')
            if self.no_improve >= self.patience:
                print('  [ES] Paciencia agotada. Parando entrenamiento.')
                control.should_training_stop = True


class StepLossCallback(TrainerCallback):
    '''Loguea loss, learning rate y grad_norm en cada logging step.'''
    def __init__(self): self.log = []
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            e = {'step':state.global_step,'loss':logs.get('loss'),
                 'lr':logs.get('learning_rate'),'grad_norm':logs.get('grad_norm','N/A')}
            self.log.append(e)
            gn = e['grad_norm'] if isinstance(e['grad_norm'],str) else f"{e['grad_norm']:.2f}"
            print(f"  [step {e['step']}] loss={e['loss']:.4f} lr={e['lr']:.2e} grad={gn}")

print('Callbacks OK')

In [ ]:
# --- Celda de prueba del CollapseCallback ---
print('Probando CollapseCallback...')
test_cb = CollapseCallback(val_df, processor, device, check_every=1)

class FakeState:
    global_step = 1

class FakeArgs:
    pass

test_cb.on_step_end(FakeArgs(), FakeState(), None, model=model)
print('Prueba OK')

Probando CollapseCallback...

  [step 1] dist={'4': 12, '1': 1, 'UNKNOWN': 2} | COLAPSO -> "4" 80%
  Acc por clase: {'1': '0/3', '2': '0/3', '3': '0/3', '4': '3/3', '5': '0/3'}
  --- Outputs (primeros 5) ---
  GT birads=1 density=C
  PRED: ```json
{"density": "B", "findings": "A 0.8 cm, irregular, slightly spiculated density is seen in the right breast, CC view. No findings are seen in the left breast, CC view. A 0.6 cm, irregular, slig
  -> birads parsed=4 FAIL

  GT birads=1 density=D
  PRED: ```json
{"density": "B", "findings": "A 1.0 cm mass is seen in the right breast, CC view. A 0.8 cm mass is seen in the left breast, MLO view.", "birads": "4"}
```
  -> birads parsed=4 FAIL

  GT birads=1 density=C
  PRED: ```json
{
  "density": "B",
  "findings": "A 0.8 cm, slightly irregular, non-mass-like density is seen in the right breast, CC view. A 0.6 cm, slightly irregular, non-mass-like density is seen in the 
  -> birads parsed=4 FAIL

  GT birads=2 density=C
  PRED: ```json
{"density"

## 14. Baseline sin fine-tuning

Evaluar el modelo base ANTES de cualquier fine-tuning para establecer el punto de partida.
MedGemma base sin fine-tuning en VinDr tipicamente logra ~20% acc en BIRADS y ~13% en density.
Esto confirma que el formato 2x2 y el dominio especifico requieren fine-tuning.

In [ ]:
print('=== BASELINE SIN FINE-TUNING ===')
print('(esperado: ~20% BIRADS acc, ~13% density acc)')
baseline_results = validate_custom(model, val_df, processor, device,
                                   n_per_class=3, max_new_tokens=124, show_examples=5)

=== BASELINE SIN FINE-TUNING ===
(esperado: ~20% BIRADS acc, ~13% density acc)

GT birads=1 density=C
OUTPUT: '```json\n{"density": "C", "findings": "Multiple clustered microcalcifications found in the right breast, CC view. Multiple clustered microcalcifications found in the left breast, CC view. Architectural distortion is present in both breasts. The right breast shows a possible asymmetry in the upper outer quadrant. The left breast shows a possible asymmetry in the upper outer quadrant.", "birads": "4"}\n```'
Parsed -> birads=4 FAIL | density=C OK

GT birads=1 density=C
OUTPUT: '```json\n{"density": "B", "findings": "A 1.0 cm mass is seen in the right breast, CC view. A 0.8 cm suspicious calcification is seen in the right breast, CC view. A 0.6 cm suspicious calcification is seen in the left breast, CC view. Architectural distortion is seen in the right breast, MLO view.", "birads": "4"}\n```'
Parsed -> birads=4 FAIL | density=B FAIL

GT birads=1 density=C
OUTPUT: '```json\n{"dens

KeyboardInterrupt: 

## 15. Training principal — Run 1 (replica receta MammoWise)

**Objetivo:** cerrar el gap a MammoWise (~0.60 acc; hoy 0.489) con una corrida limpia y medida.

**Receta:**
- Target **descriptivo** (birads y density con descripcion + findings) — empiricamente mejor que el corto
- `lr=2e-4`, `r=16/alpha=16` (scaling 1.0), 10 epocas, scheduler linear, warmup 0.03
- LoRA en **todos los linear del LM** (attn + FFN), vision encoder congelado
- **Masking del prompt** (assistant-only) + CE estandar (sin loss custom)

Checkpoints por epoca; evaluar todos sobre los 1000 oficiales y elegir por BIRADS F1 macro.

In [ ]:
import gc
# Limpiar VRAM antes de training
torch.cuda.empty_cache()
gc.collect()
model.train()
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
print(f'VRAM antes de training: {torch.cuda.memory_allocated()/1e9:.1f}GB')
print('Listo para training')

VRAM antes de training: 4.8GB
Listo para training


In [ ]:
# Run 5b: MultiTaskCDWTrainer desactivado
# CDW creaba atractor en BIRADS 3 (centro ordinal). Sustituido por label_smoothing.
print('CDW desactivado - usando SFTTrainer estandar')

In [ ]:
import os, gc
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
from trl import SFTConfig


# Limpiar VRAM
torch.cuda.empty_cache(); gc.collect()
model.train()
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})

step_cb     = StepLossCallback()
collapse_cb = CollapseCallback(
    val_df, processor, device,
    check_every=200,
    es_patience=CONFIG['es_patience'],
)

sft_cfg = SFTConfig(
    output_dir                    = str(CHECKPOINT_DIR / 'run'),
    num_train_epochs              = CONFIG['epochs'],
    per_device_train_batch_size   = CONFIG['batch_size'],
    per_device_eval_batch_size    = 1,
    gradient_accumulation_steps   = CONFIG['grad_accum'],
    gradient_checkpointing        = True,
    gradient_checkpointing_kwargs = {'use_reentrant': False},
    optim                         = 'adamw_bnb_8bit',
    bf16                          = True,
    fp16                          = False,
    logging_steps                 = 25,
    eval_strategy                 = 'no',
    save_strategy                 = 'epoch',
    save_total_limit              = 15,
    load_best_model_at_end        = False,
    learning_rate                 = CONFIG['lr'],
    lr_scheduler_type             = 'linear',          # Run 6: MammoWise paper/codigo
    warmup_ratio                  = CONFIG['warmup_ratio'],
    max_grad_norm                 = CONFIG['max_grad_norm'],
    weight_decay                  = CONFIG['weight_decay'],
    label_smoothing_factor        = CONFIG['label_smoothing'],
    report_to                     = 'none',
    dataset_kwargs                = {'skip_prepare_dataset': True},
    remove_unused_columns         = False,
    label_names                   = ['labels'],
    seed                          = CONFIG['seed'],
)

trainer = SFTTrainer(
    model            = model,
    args             = sft_cfg,
    train_dataset    = train_ds,
    eval_dataset     = val_ds,
    processing_class = processor,
    data_collator    = collate,
    callbacks        = [step_cb, collapse_cb],
)

steps_per_epoch = len(train_ds) // (CONFIG['batch_size'] * CONFIG['grad_accum'])
print(f'Steps/epoca: {steps_per_epoch} | Total max: ~{steps_per_epoch * CONFIG["epochs"]}')
print(f'LR={CONFIG["lr"]} | r={CONFIG["lora_r"]} alpha={CONFIG["lora_alpha"]} | weight_decay={CONFIG["weight_decay"]}')
print(f'label_smoothing={CONFIG["label_smoothing"]} | ES patience={CONFIG["es_patience"]}')

trainer.train()
print('\nEntrenamiento completado')

import json as _j
out_dir = CHECKPOINT_DIR / 'run'
with open(out_dir / 'step_logs.json', 'w') as f:
    _j.dump(step_cb.log, f, indent=2)
with open(out_dir / 'collapse_logs.json', 'w') as f:
    _j.dump(collapse_cb.history, f, indent=2)
print(f'Logs guardados en {out_dir}')

## 16. Evaluacion de checkpoints

Evaluar cada checkpoint guardado durante training para seleccionar el mejor segun BIRADS F1 macro.
Se usa `validate_custom` con el val set completo (n_per_class=1000 = todos los disponibles).

**Por que seleccionar por F1 macro y no accuracy:** F1 macro pondera igualmente todas las clases, penalizando modelos que ignoran BIRADS 3/4/5 minoritarios.

In [ ]:
from transformers import AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel

# Evaluar todos los checkpoints disponibles
ckpt_dir = CHECKPOINT_DIR / 'run'
checkpoints = sorted(ckpt_dir.glob('checkpoint-*'), key=lambda x: int(x.name.split('-')[1]))
print(f'Checkpoints encontrados: {len(checkpoints)}')
for ck in checkpoints:
    print(f'  {ck.name}')

all_results = []

for ckpt in checkpoints:
    print(f'\n=== Evaluando {ckpt.name} ===')

    # Recargar modelo base + checkpoint
    model_eval = AutoModelForImageTextToText.from_pretrained(
        CONFIG['model_id'],
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
        ),
        device_map='auto', torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True, attn_implementation='eager',
    )
    model_eval = PeftModel.from_pretrained(model_eval, str(ckpt))
    model_eval.eval()

    results = validate_custom(
        model_eval, val_df, processor, device,
        n_per_class=1000,  # todos los disponibles en val
        max_new_tokens=124, show_examples=0
    )
    results['checkpoint'] = ckpt.name
    all_results.append(results)

    del model_eval; torch.cuda.empty_cache(); gc.collect()

# Resumen final
print('\n=== RESUMEN DE CHECKPOINTS ===')
print(f'{"Checkpoint":<25} {"BIRADS acc":>12} {"BIRADS F1":>10} {"Density acc":>12} {"Density F1":>10}')
best_ckpt = max(all_results, key=lambda x: x['birads_f1'])
for r in all_results:
    marker = ' <-- MEJOR' if r['checkpoint'] == best_ckpt['checkpoint'] else ''
    print(f'{r["checkpoint"]:<25} {r["birads_accuracy"]:>12.4f} {r["birads_f1"]:>10.4f} '
          f'{r["density_accuracy"]:>12.4f} {r["density_f1"]:>10.4f}{marker}')

print(f'\nMejor checkpoint por BIRADS F1: {best_ckpt["checkpoint"]}')